In [ ]:
# ============================================================
# LIGHTGBM — MEDIUM DEPTH 🟡
# ============================================================
#
# No scratch implementation needed.
# Learn:
#   1. Why LightGBM
#   2. Internal working
#   3. Histogram-based splitting
#   4. Leaf-wise growth
#   5. Important parameters
#   6. sklearn-style implementation
#   7. Experiments
#   8. Interview questions
#
# Install if needed:
# pip install lightgbm
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from lightgbm import LGBMClassifier


# ============================================================
# 2. WHY LIGHTGBM?
# ============================================================
#
# LightGBM is a Gradient Boosting algorithm based on
# decision trees.
#
# Basic idea:
#
# Initial prediction
#       ↓
# Calculate error / gradient
#       ↓
# Build tree to reduce error
#       ↓
# Add tree
#       ↓
# Repeat
#
# LightGBM makes Gradient Boosting faster using:
#
# 1. Histogram-based split finding
# 2. Leaf-wise tree growth
# 3. GOSS
# 4. EFB
#
# Main use case:
# Large tabular datasets
#
# ============================================================


# ============================================================
# 3. CREATE DATASET
# ============================================================

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


# ============================================================
# 4. BASIC LIGHTGBM MODEL
# ============================================================

model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))


# ============================================================
# 5. INTERNAL WORKING
# ============================================================
#
# LightGBM is still Gradient Boosting.
#
# General boosting formula:
#
# F_m(x) = F_(m-1)(x) + learning_rate * tree_m(x)
#
# At every iteration:
#
# Previous model
#       ↓
# Calculate gradients
#       ↓
# Calculate Hessians
#       ↓
# Find useful split
#       ↓
# Grow tree
#       ↓
# Add tree to model
#
# Important difference:
#
# Traditional tree growth:
#
#            Root
#           /    \
#          A      B
#         / \    / \
#        C   D  E   F
#
# Level-wise:
# All nodes at the same level are expanded.
#
#
# LightGBM:
#
# It checks available leaves and chooses the leaf
# with the highest split gain.
#
# Example:
#
# Leaf A -> gain = 5
# Leaf B -> gain = 15
# Leaf C -> gain = 3
#
# LightGBM chooses:
#
# Leaf B
#
# because:
#
# highest gain = 15
#
#
# This is called:
#
# LEAF-WISE TREE GROWTH
#
# ============================================================


# ============================================================
# 6. HISTOGRAM-BASED SPLITTING
# ============================================================
#
# Suppose a feature contains:
#
# 10.1, 10.2, 10.3, 10.5, 20.1, 20.4, 30.2
#
# Instead of considering every exact value as a split,
# LightGBM can put values into bins:
#
# 0 - 10
# 10 - 20
# 20 - 30
# 30 - 40
#
# Example:
#
# 10.1 -> Bin 2
# 10.2 -> Bin 2
# 10.3 -> Bin 2
# 20.1 -> Bin 3
# 20.4 -> Bin 3
# 30.2 -> Bin 4
#
# Then LightGBM searches splits between bins.
#
# Fewer split candidates
#       ↓
# Less computation
#       ↓
# Faster training
#       ↓
# Lower memory usage
#
# ============================================================


# ============================================================
# 7. GRADIENT + HESSIAN
# ============================================================
#
# For each training example:
#
# Gradient:
#
# g_i = dL / dF(x_i)
#
# Hessian:
#
# h_i = d²L / dF(x_i)²
#
# LightGBM uses these values to calculate how useful
# a potential split is.
#
# You DON'T need to implement this from scratch.
#
# Just remember:
#
# Gradient -> direction of error
# Hessian  -> curvature of loss
#
# Gradient + Hessian
#        ↓
# Evaluate splits
#        ↓
# Select best split
#
# ============================================================


# ============================================================
# 8. IMPORTANT PARAMETERS
# ============================================================

# ------------------------------------------------------------
# n_estimators
# ------------------------------------------------------------
#
# Number of boosting trees.
#
# Higher:
#   More model capacity
#   More training time
#   Possible overfitting
#
#
# ------------------------------------------------------------
# learning_rate
# ------------------------------------------------------------
#
# Contribution of each tree.
#
# Smaller learning_rate:
#   Smaller updates
#   Usually need more trees
#
#
# ------------------------------------------------------------
# num_leaves ⭐⭐⭐
# ------------------------------------------------------------
#
# Maximum number of leaves in a tree.
#
# Very important LightGBM parameter.
#
# Higher:
#   More complex trees
#   More flexible model
#   Higher overfitting risk
#
#
# ------------------------------------------------------------
# max_depth
# ------------------------------------------------------------
#
# Maximum tree depth.
#
# -1 means no explicit limit.
#
# Useful for controlling very deep trees.
#
#
# ------------------------------------------------------------
# min_child_samples
# ------------------------------------------------------------
#
# Minimum number of samples required in a leaf.
#
# Higher:
#   Larger leaves
#   Simpler model
#   Less overfitting
#
#
# ------------------------------------------------------------
# subsample
# ------------------------------------------------------------
#
# Fraction of rows used.
#
# Example:
#
# subsample=0.8
#
# approximately 80% of training rows.
#
# In LightGBM, use subsample_freq > 0 to activate
# row subsampling.
#
#
# ------------------------------------------------------------
# subsample_freq
# ------------------------------------------------------------
#
# Frequency of row subsampling.
#
# Example:
#
# subsample_freq=1
#
#
# ------------------------------------------------------------
# colsample_bytree
# ------------------------------------------------------------
#
# Fraction of features used for each tree.
#
# Example:
#
# colsample_bytree=0.8
#
# approximately 80% of features.
#
#
# ------------------------------------------------------------
# reg_alpha
# ------------------------------------------------------------
#
# L1 regularization.
#
#
# ------------------------------------------------------------
# reg_lambda
# ------------------------------------------------------------
#
# L2 regularization.
#
#
# ------------------------------------------------------------
# min_split_gain
# ------------------------------------------------------------
#
# Minimum gain required to perform a split.
#
# Higher:
#   Fewer splits
#   Simpler model
#
# ============================================================


# ============================================================
# 9. MODEL WITH IMPORTANT PARAMETERS
# ============================================================

model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,

    # Tree complexity
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,

    # Row / feature sampling
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    # Regularization
    reg_alpha=0.0,
    reg_lambda=1.0,

    # Minimum gain required for a split
    min_split_gain=0.0,

    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, pred))


# ============================================================
# 10. EXPERIMENT — NUM_LEAVES
# ============================================================
#
# num_leaves controls tree complexity.
#
# Let's see what happens when it changes.
# ============================================================

leaf_values = [7, 15, 31, 63, 127]

train_scores = []
test_scores = []

for leaves in leaf_values:

    model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=leaves,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_scores.append(
        accuracy_score(y_train, train_pred)
    )

    test_scores.append(
        accuracy_score(y_test, test_pred)
    )


plt.figure(figsize=(8, 5))

plt.plot(
    leaf_values,
    train_scores,
    marker="o",
    label="Train"
)

plt.plot(
    leaf_values,
    test_scores,
    marker="o",
    label="Test"
)

plt.xlabel("num_leaves")
plt.ylabel("Accuracy")
plt.title("Effect of num_leaves")

plt.legend()
plt.show()


# ============================================================
# 11. FEATURE IMPORTANCE
# ============================================================

importance = model.feature_importances_

print("\nFeature Importance:")

for i, value in enumerate(importance):
    print(f"Feature {i}: {value}")


plt.figure(figsize=(8, 5))

plt.bar(
    range(len(importance)),
    importance
)

plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("LightGBM Feature Importance")

plt.show()


# ============================================================
# 12. LIGHTGBM vs OTHER BOOSTING ALGORITHMS
# ============================================================
#
# Gradient Boosting:
#
# Gradient / residual
#       ↓
# Train tree
#       ↓
# Add tree
#       ↓
# Repeat
#
#
# XGBoost:
#
# Gradient + Hessian
#       ↓
# Find splits
#       ↓
# Regularization
#       ↓
# Add tree
#
#
# LightGBM:
#
# Gradient + Hessian
#       ↓
# Histogram-based split finding
#       ↓
# Leaf with highest gain
#       ↓
# Leaf-wise growth
#       ↓
# Add tree
#
#
# ============================================================


# ============================================================
# 13. LIGHTGBM vs RANDOM FOREST
# ============================================================
#
# Random Forest:
#
# Tree 1 ──┐
# Tree 2 ──┤
# Tree 3 ──┤──> Majority Vote
# Tree 4 ──┤
# Tree 5 ──┘
#
# Trees are mostly independent.
#
#
# LightGBM:
#
# Tree 1
#   ↓
# Tree 2 fixes previous errors
#   ↓
# Tree 3 fixes previous errors
#   ↓
# Tree 4
#
# Trees are sequential.
#
#
# Random Forest = Bagging
# LightGBM     = Boosting
#
# ============================================================


# ============================================================
# 14. IMPORTANT LIGHTGBM OPTIMIZATIONS
# ============================================================
#
# 1. Histogram-based splitting
#    → faster split search
#
# 2. Leaf-wise tree growth
#    → split highest-gain leaf
#
# 3. GOSS
#    → Gradient-based One-Side Sampling
#    → keeps more important high-gradient examples
#
# 4. EFB
#    → Exclusive Feature Bundling
#    → combines sparse/exclusive features
#
# ============================================================


# ============================================================
# 15. WHEN TO USE LIGHTGBM
# ============================================================
#
# Good for:
#
# - Tabular data
# - Large datasets
# - Classification
# - Regression
# - Ranking
# - Many features
#
#
# Usually not the first choice for:
#
# - Raw images
# - Raw audio
# - Raw text
#
# For those, neural networks / transformers are often
# more appropriate.
#
# ============================================================


# ============================================================
# 16. INTERVIEW QUESTIONS
# ============================================================
#
# Q1. What is LightGBM?
#
# Answer:
# LightGBM is a gradient boosting framework based on
# decision trees, optimized for efficient training using
# histogram-based split finding and leaf-wise tree growth.
#
#
# Q2. What is leaf-wise growth?
#
# Answer:
# Instead of expanding all nodes level by level, LightGBM
# selects the leaf with the highest split gain and expands
# that leaf.
#
#
# Q3. Why can LightGBM overfit?
#
# Answer:
# Leaf-wise growth can create very complex trees.
# Parameters such as num_leaves, max_depth and
# min_child_samples help control complexity.
#
#
# Q4. Why is LightGBM fast?
#
# Answer:
# Mainly because of histogram-based split finding,
# efficient memory usage and optimizations such as GOSS
# and EFB.
#
#
# Q5. What is num_leaves?
#
# Answer:
# It controls the maximum number of leaves in a tree
# and is one of the most important parameters controlling
# LightGBM model complexity.
#
#
# Q6. LightGBM vs Random Forest?
#
# Answer:
# Random Forest uses bagging and mostly independent trees.
# LightGBM uses boosting where trees are built sequentially
# to correct previous errors.
#
#
# ============================================================
# 17. 30-SECOND INTERVIEW EXPLANATION
# ============================================================
#
# LightGBM is a gradient boosting tree algorithm designed
# for efficient training on large tabular datasets.
#
# It uses histogram-based split finding to reduce the number
# of candidate split points and grows trees leaf-wise by
# selecting the leaf with the highest gain.
#
# This makes it fast and powerful, but leaf-wise growth can
# overfit, so parameters such as num_leaves, max_depth and
# min_child_samples are important for controlling complexity.
#
# ============================================================


# ============================================================
# 18. FINAL MEMORY CHEATSHEET
# ============================================================
#
# LightGBM =
#
# Gradient Boosting
#       +
# Histogram-based splitting
#       +
# Leaf-wise growth
#
#
# MUST REMEMBER:
#
# 1. Gradient Boosting
# 2. Gradient + Hessian
# 3. Histogram-based split finding
# 4. Leaf-wise growth
# 5. num_leaves ⭐
# 6. Leaf-wise growth can overfit
# 7. Control using:
#       num_leaves
#       max_depth
#       min_child_samples
#
# ============================================================